# v11


In [ ]:
# Step 1: YandexGPT credentials
import os, requests
os.environ.setdefault('YANDEX_API_KEY',   'AQVN12VhDjMgke12NrKtdwr05MgZwMKEbHiNRKBt')
os.environ.setdefault('YANDEX_FOLDER_ID', 'b1ggp14rml9bbc37v93l')
os.environ.setdefault('YANDEX_MODEL',     'yandexgpt')
api_key = os.environ['YANDEX_API_KEY']
folder_id = os.environ['YANDEX_FOLDER_ID']
print(f'[ENV] KEY={api_key[:8]}..., FOLDER={folder_id}')
headers = {'Authorization': f'Api-Key {api_key}', 'Content-Type': 'application/json'}
payload = {'modelUri': f'gpt://{folder_id}/yandexgpt/latest',
           'completionOptions': {'stream': False, 'temperature': 0.1, 'maxTokens': '10'},
           'messages': [{'role': 'user', 'text': 'Say OK'}]}
try:
    r = requests.post('https://llm.api.cloud.yandex.net/foundationModels/v1/completion',
                      headers=headers, json=payload, timeout=15)
    r.raise_for_status()
    print(f'[LLM PING] {r.json()["result"]["alternatives"][0]["message"]["text"]}')
except Exception as e:
    print(f'[LLM PING] ERROR: {e}')

In [ ]:
# Step 2: Load KG (full.txt) → fast dict index
import sys, os, time
from collections import defaultdict
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path: sys.path.insert(0, project_root)
from src.utils.data_structs import QuadrupletCreator, NodeCreator, RelationCreator, RelationType, NodeType
kg_path = os.path.join(project_root, 'wikidata_big/kg')

print('[MAP] Loading id->name mappings...')
ent_map, rel_map = {}, {}
with open(f'{kg_path}/wd_id2entity_text.txt') as f:
    for line in f:
        p = line.strip().split('\t')
        if len(p) >= 2: ent_map[p[0]] = p[1]
with open(f'{kg_path}/wd_id2relation_text.txt') as f:
    for line in f:
        p = line.strip().split('\t')
        if len(p) >= 2: rel_map[p[0]] = p[1]
print(f'[MAP] {len(ent_map):,} entities, {len(rel_map):,} relations')

print('[GRAPH] Parsing full.txt...')
t0 = time.time()
quads_raw, errors, seen_ids = [], 0, set()
with open(f'{kg_path}/full.txt') as f:
    for line in f:
        p = line.strip().split('\t')
        if len(p) < 3: continue
        try:
            t_name = f'{p[3]} - {p[4]}' if len(p) > 4 else (p[3] if len(p) > 3 else 'Always')
            s_node = NodeCreator.create(NodeType.object, ent_map.get(p[0], p[0]), prop={'wd_id': p[0]})
            r_node = RelationCreator.create(RelationType.simple, name=rel_map.get(p[1], p[1]), prop={'wd_id': p[1]})
            o_node = NodeCreator.create(NodeType.object, ent_map.get(p[2], p[2]), prop={'wd_id': p[2]})
            t_node = NodeCreator.create(NodeType.time, t_name, add_stringified_node=True)
            quad = QuadrupletCreator.create(s_node, r_node, o_node, t_node)
            if quad.id not in seen_ids:
                quads_raw.append(quad); seen_ids.add(quad.id)
        except: errors += 1
print(f'[GRAPH] {len(quads_raw):,} quads in {time.time()-t0:.1f}s (err={errors})')

print('[INDEX] Building...')
t1 = time.time()
wd_id_to_quads = defaultdict(list)
name_to_quads  = defaultdict(list)
for quad in quads_raw:
    s_wd = quad.start_node.prop.get('wd_id')
    o_wd = quad.end_node.prop.get('wd_id')
    if s_wd: wd_id_to_quads[s_wd].append(quad)
    if o_wd: wd_id_to_quads[o_wd].append(quad)
    name_to_quads[quad.start_node.name].append(quad)
    name_to_quads[quad.end_node.name].append(quad)
print(f'[INDEX] Done in {time.time()-t1:.1f}s')

In [ ]:
# Step 3: Init LLM + encoders + WikidataMapper
import torch
from sentence_transformers import SentenceTransformer
from src.llm.yandex_gpt_client import YandexGPTClient
from src.utils.wikidata_utils import WikidataMapper

llm_client = YandexGPTClient(
    api_key=os.environ['YANDEX_API_KEY'],
    folder_id=os.environ['YANDEX_FOLDER_ID'],
    model_name=os.environ.get('YANDEX_MODEL', 'yandexgpt')
)
print('[LLM] YandexGPT ready')

encoder      = SentenceTransformer('intfloat/multilingual-e5-small')  # advanced
encoder_base = SentenceTransformer('intfloat/multilingual-e5-small')  # baseline (raw)
print('[E5] encoders ready (advanced + baseline)')

mapper = WikidataMapper(kg_path)
print('[MAP] WikidataMapper ready')

temporal_scorer = None
try:
    from src.kg_model.temporal.temporal_model import TemporalScorer
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    temporal_scorer = TemporalScorer(device=device)
    print(f'[TComplEx] Loaded')
except Exception as e:
    print(f'[TComplEx] Skipped: {e}')

print('\nAll systems ready!')

In [ ]:
# Step 4: Final QA Logic (Advanced + Baseline)
import numpy as np
import time
import re
import json

# --- Helper Functions ---
def log_debug(title, content, show=False):
    """Prints a formatted debug block."""
    if not show: return
    try:
        if isinstance(content, str):
            c_str = content
        else:
            c_str = json.dumps(content, indent=2, ensure_ascii=False)
        print(f'\n[DBG] {title}\n{"-"*40}\n{c_str}\n{"-"*40}')
    except:
        print(f'\n[DBG] {title}\n{"-"*40}\n{content}\n{"-"*40}')

def resolve_entity(name: str):
    """3-step entity resolution: exact match -> fuzzy KG search -> LLM normalization."""
    if not name: return None, None

    # Step 1: exact match
    wd_id = mapper.get_id(name)
    if wd_id: return name, wd_id

    # Step 2: fuzzy search in KG names, LLM picks best
    cands = mapper.search_names(name, limit=10)
    if not cands and ' ' in name:
        cands = mapper.search_names(name.split()[0], limit=10)
    if cands:
        if len(cands) == 1:
            return cands[0], mapper.get_id(cands[0])
        cand_str = ', '.join(repr(c) for c in cands)
        prompt = (f"A question mentions entity '{name}'. "
                  f"Which of these KG names best matches?\nOptions: {cand_str}\n"
                  f"Output ONLY the exact name from the list, nothing else.")
        try:
            choice = llm_client.generate(prompt).strip().strip("\"'")
            if choice in cands: return choice, mapper.get_id(choice)
        except: pass
        return cands[0], mapper.get_id(cands[0])

    # Step 3: LLM normalization (e.g. 'USA' -> 'United States of America')
    try:
        norm = llm_client.generate(
            f"What is the full official Wikidata/Wikipedia name of '{name}'? "
            f"Output ONLY the name, nothing else."
        ).strip().strip("\"'")
        wd_id = mapper.get_id(norm)
        if wd_id: return norm, wd_id
        cands2 = mapper.search_names(norm, limit=5)
        if cands2: return cands2[0], mapper.get_id(cands2[0])
    except: pass

    return name, None

def parse_years(quad):
    """Extracts (start_year, end_year) from a quad."""
    if not quad.time or quad.time.name == 'Always': return None, None
    m = re.findall(r'(\d{4})', quad.time.name)
    if not m: return None, None
    years = sorted([int(y) for y in m])
    return years[0], years[-1]

def build_anon_ctx(quads):
    """Build anonymized context using Wikidata Q/P IDs only."""
    lines, seen = [], set()
    for q in quads:
        s = q.start_node.prop.get('wd_id', '?')
        r = q.relation.prop.get('wd_id', '?')
        o = q.end_node.prop.get('wd_id', '?')
        t = q.time.name if q.time else 'Always'
        sig = f'{s}-{r}-{o}-{t}'
        if sig not in seen:
            seen.add(sig)
            lines.append(f'- {s} --[{r}]--> {o} (Time: {t})')
    return '\n'.join(lines)

def decode_ans(raw_ans):
    """Extracts Q-ID from LLM response."""
    m = re.search(r'([QP]\d+)', raw_ans)
    return m.group(1) if m else raw_ans

# Generalized System Prompt (B8 Fix)
ANON_SYS = "You are a pure logical reasoning engine. To answer the question, identify the relevant fact among the provided IDs. FACT STRUCTURE: 'Subject_ID --[Relation_ID]--> Object_ID (Time: Range)' Identify which entity ID (Subject or Object) is the specific placeholder requested by the question. For example, if asked for a 'capital of X', identify whether the Subject or Object acts as the name of the city. Output ONLY the ID (e.g. Q123 or P123). If no answer: NULL." from the FACTS below. "
    "FACT STRUCTURE: 'Subject_ID --[Relation_ID]--> Object_ID (Time: Range)' "
    "To answer the question, identify the relevant fact and choose the ID that "
    "represents the specific value being asked for (the target of the relation). "
    "Do NOT use external knowledge. If the answer is not in facts: output NULL."
)

# --- Main Functions ---

def map_id_to_name(qid):
    if qid.startswith('Q'):
        return f"{ent_map.get(qid, qid)} ({qid})"
    if qid.startswith('P'):
        return f"{rel_map.get(qid, qid)} ({qid})"
    return qid

def ask(question: str, top_k: int = 10, anonymize: bool = True, debug: bool = False):
    SEP = '=' * 80
    print(f'\n{SEP}\nQUESTION: {question}\n{SEP}')
    t0 = time.time()

    # 1. Extraction
    print('\n[1/4] Extracting...')
    ext = llm_client.extract_search_parameters(question)
    log_debug('EXTRACTION', ext, show=debug)
    
    entities     = ext.get('entities', [])
    query_time   = ext.get('time')
    anchor_ent   = ext.get('anchor_entity')
    anchor_event = ext.get('anchor_event')
    q_type       = ext.get('type', 'simple_entity')
    
    # 2. Config
    alpha = {'simple_time': 0.6, 'before_after': 0.45, 'time_join': 0.5, 'first_last': 0.5}.get(q_type, 0.3)
    filter_temporal = q_type in ('before_after', 'time_join')
    search_k = 50 if filter_temporal or q_type in ('simple_time', 'first_last') else 15
    print(f'  Type={q_type} alpha={alpha} search_k={search_k}')

    # 3. Step 2: HOP 1 (find anchor's attribute time)
    resolved_time = query_time
    if anchor_ent and anchor_event and not query_time and q_type != 'simple_time':
         print(f'\n[2/4] HOP 1: Resolving time for "{anchor_event}" of "{anchor_ent}"...')
         rn, aw = resolve_entity(anchor_ent)
         if aw:
             hq = wd_id_to_quads.get(aw, [])[:15]
             ctx = '\n'.join(QuadrupletCreator.stringify(q)[1] for q in hq)
             prompt = f"FACTS:\n{ctx}\n\nExtract ONLY the 4-digit YEAR for '{anchor_event}' of '{rn}':"
             try:
                 raw = llm_client.generate(prompt).strip()
                 m = re.search(r'(\d{4})', raw)
                 if m: 
                     resolved_time = m.group(1)
                     print(f'  -> Hop 1 resolved year: {resolved_time}')
             except: pass
    
    # 4. Retrieval & Buckets
    search_time = resolved_time
    print(f'\n[3/4] Retrieving (time={search_time})...')
    candidates_raw = []
    for ent in entities:
        res_name, wd_id = resolve_entity(ent)
        batch = wd_id_to_quads.get(wd_id, []) if wd_id else name_to_quads.get(ent, [])
        candidates_raw.extend(batch)
        print(f'  - {len(batch)} quads for "{res_name}"')

    # FIX B3+B4: Pre-filter for before_after
    if q_type == 'before_after' and search_time:
        try:
            ref = int(search_time)
            is_before = 'before' in question.lower()
            filtered = [q for q in candidates_raw if (parse_years(q)[0] is not None and 
                        (is_before and parse_years(q)[0] < ref or not is_before and parse_years(q)[1] > ref))]
            if filtered:
                candidates_raw = filtered
                print(f'  [B3] pre-filter applied: {len(filtered)} quads')
        except: pass

    unique, seen = [], set()
    for q in candidates_raw:
        if q.id not in seen: unique.append(q); seen.add(q.id)
    if not unique:
        print('>>> NO CANDIDATES'); return 'Unknown'

    # FIX B1+B2: Temporal bucket
    temporal_bucket = []
    if q_type == 'time_join' and search_time:
        try:
            t_int = int(search_time)
            seen_b = set()
            for ent in entities:
                _, wd_id = resolve_entity(ent)
                all_q = wd_id_to_quads.get(wd_id, []) if wd_id else name_to_quads.get(ent, [])
                for q in all_q:
                    sy, ey = parse_years(q)
                    if sy is not None and sy <= t_int <= ey and q.id not in seen_b:
                        temporal_bucket.append(q); seen_b.add(q.id)
            print(f'  [B1/B2] temporal_bucket: {len(temporal_bucket)} quads @ {search_time}')
            search_k = max(search_k, 20)
        except: pass

    # Scoring
    q_emb = encoder.encode(['query: ' + question])[0]
    txts = [QuadrupletCreator.stringify(q)[1] for q in unique]
    embs = encoder.encode(['passage: ' + t for t in txts])
    results = []
    for quad, text, emb in zip(unique, txts, embs):
        e5 = float(np.dot(q_emb, emb) / (np.linalg.norm(q_emb)*np.linalg.norm(emb) + 1e-9))
        tp, tl, final = 0.0, -100.0, e5
        if search_time and temporal_scorer:
             sid, rid, oid = quad.start_node.prop.get('wd_id'), quad.relation.prop.get('wd_id'), quad.end_node.prop.get('wd_id')
             if sid and rid and oid:
                 try:
                     tl = float(temporal_scorer.score(sid, rid, oid, str(search_time)))
                     tp = float(1/(1+np.exp(-tl))); final = (1-alpha)*e5 + alpha*tp
                 except: pass
        if filter_temporal and quad.time and quad.time.name != 'Always': final += 1.0
        results.append({'text': text, 'quad': quad, 'conf': final, 'e5': e5, 'tp': tp, 'tl': tl})
    results.sort(key=lambda x: x['conf'], reverse=True)
    top = results[:search_k]

    log_debug('TOP RESULTS', '\n'.join(f"[{r['conf']:.3f}] E5={r['e5']:.3f} T={r['tp']:.3f} | {r['text']}" for r in top[:10]), show=debug)

    # FIX B7: first_last
    if q_type == 'first_last':
        timed = [(parse_years(r['quad'])[0], r) for r in top if parse_years(r['quad'])[0] is not None]
        if timed:
            timed.sort(key=lambda x: x[0])
            isf = any(w in question.lower() for w in ('first', 'earliest', 'oldest', 'initial'))
            top = [c[1] for c in (timed[:5] if isf else timed[-5:])]

    # --- Step 5: Selection ---
    all_context = temporal_bucket + [r['quad'] for r in top]
    selected_quads = all_context
    if anonymize and len(all_context) > 5:
        print(f'  [B5] Selection Stage: filtering {len(all_context)} items...')
        sel_facts = []
        for i, q in enumerate(all_context[:30], 1):
            s, r, o = q.start_node.name, q.relation.name, q.end_node.name
            t = q.time.name if q.time else 'Always'
            sel_facts.append(f"[{i}] [{t}] {s} --[{r}]--> {o}")
        
        sel_prompt = f"QUESTION: {question}\n\nFACTS:\n" + "\n".join(sel_facts) + (
            "\n\nSelect 3-5 fact NUMBERS that best help answer the question (comma-separated). "
            "If none, say NONE."
        )
        try:
            resp = llm_client.generate(sel_prompt).strip()
            nums = [int(n) for n in re.findall(r'(\d+)', resp)]
            if nums: 
                selected_quads = [all_context[n-1] for n in nums if 0 < n <= len(all_context)]
                print(f'  [B5] Selected {len(selected_quads)} facts.')
        except: pass

    # --- Step 6: Final Answer ---
    print('\n[4/4] Generating answer...')
    if anonymize:
        ctx = build_anon_ctx(selected_quads)
        um = f'QUESTION: {question}\nTIME CONTEXT: {search_time}\nFACTS:\n{ctx}\nANSWER (Q-ID only):'
        log_debug('PROMPT (ANON)', f'SYS: {ANON_SYS[:80]}...\nUSER: {um}', show=debug)
        try:
            qid = decode_ans(llm_client.generate(um, system=ANON_SYS).strip())
            ans = map_id_to_name(qid)
        except: ans = 'Error'
    else:
        ctx = '\n'.join(f"- {r['text']}" for r in top)
        um = f'QUESTION: {question}\nFACTS:\n{ctx}\nANSWER:'
        try: ans = llm_client.generate(um, system="Answer concise Name only.").strip()
        except: ans = 'Error'

    print(f'\n{SEP}\n>>> FINAL ANSWER: {ans} ({time.time()-t0:.2f}s)\n{SEP}\n')
    return ans

def dbg(question, **kwargs):
    return ask(question, debug=True, **kwargs)

def ask_base(question: str, top_k: int = 10, anonymize: bool = True):
    """
    Pure Baseline (v2 alignment).
    - Strict Exact Match Entity Selection ONLY (no LLM fuzzy search).
    - Raw E5 ranking ONLY (no TComplEx, no temporal buckets).
    - Mandatory Anonymization (Q-ID mapping) to prevent LLM hallucination.
    """
    SEP = '=' * 80
    print(f'\n{SEP}\n[BASE] QUESTION: {question}\n{SEP}')
    t0 = time.time()
    
    # 1. Extraction (Using standard LLM extractor for fairness to get entities)
    ext = llm_client.extract_search_parameters(question)
    entities = ext.get('entities', [])
    print(f'  [1/4] Extracted Entities: {entities}')
    
    # 2. Strict V2 Retrieval (No fuzzy, no temporal buckets)
    craw = []
    for ent in entities:
        wid = mapper.get_id(ent)
        if wid: 
            batch = wd_id_to_quads.get(wid, [])
            craw.extend(batch)
            print(f'  - Retrieved {len(batch)} quads for "{ent}" (Exact Match: {wid})')
        else:
            # Fallback to string exact match (v2 behavior)
            batch = name_to_quads.get(ent, [])
            craw.extend(batch)
            print(f'  - Retrieved {len(batch)} quads for "{ent}" (String Match)')
            
    unique, seen = [], set()
    for q in craw:
        if q.id not in seen: unique.append(q); seen.add(q.id)
        
    if not unique: 
        print('>>> [BASE] NO CANDIDATES FOUND.'); return 'Unknown'
        
    # 3. Pure E5 Ranking
    print(f'  [2/4] Ranking {len(unique)} candidates using RAW E5...')
    qe = encoder_base.encode(['query: ' + question])[0]
    txts = [QuadrupletCreator.stringify(q)[1] for q in unique]
    embs = encoder_base.encode(['passage: ' + t for t in txts])
    
    res = []
    for q, t, e in zip(unique, txts, embs):
        sim = float(np.dot(qe, e) / (np.linalg.norm(qe) * np.linalg.norm(e) + 1e-9))
        res.append({'q': q, 't': t, 'c': sim})
    res.sort(key=lambda x: x['c'], reverse=True)
    top = res[:top_k]
    
    # 4. Final Inference with Minimal Prompt & Anonymization
    print('  [3/4] Formatting context...')
    if anonymize:
        ctx = build_anon_ctx([r['q'] for r in top])
        sys_msg = "Answer the question based ONLY on the provided facts. Output the corresponding ID (Q-id or P-id). If no answer is found, output NULL."
        um = f'QUESTION: {question}\nFACTS:\n{ctx}\nANSWER (ID only):'
        
        print('  [4/4] Generating answer...')
        try: 
            raw_ans = llm_client.generate(um, system=sys_msg).strip()
            qid = decode_ans(raw_ans)
            ans = map_id_to_name(qid)
        except: ans = 'Error'
    else:
        ctx = '\n'.join(f"- {r['t']}" for r in top)
        sys_msg = "Answer based ONLY on the facts above. If unknown, say Unknown. Concise name only."
        um = f'QUESTION: {question}\nFACTS:\n{ctx}\nANSWER:'
        try: ans = llm_client.generate(um, system=sys_msg).strip()
        except: ans = 'Error'
    
    print(f'\n{SEP}\n>>> [BASE] FINAL ANSWER: {ans} ({time.time()-t0:.2f}s)\n{SEP}\n')
    return ans

print('ask(), ask_base() and dbg() ready.')


# QA Experiments v3 — Demo Questions
Running 7 questions to verify each type. All use `anonymize=True`.

In [ ]:
print('\n=== Q1: simple_entity  (in_kg=True) ===')
ask('Who was the spouse of Vladimir Nabokov?')

In [ ]:
print('\n=== Q2: simple_time  (in_kg=False — expects NULL/Unknown) ===')
ask('When was Vladimir Nabokov born?')

In [ ]:
print('\n=== Q3: before_after  (in_kg=True) ===')
ask('Where did Vladimir Nabokov live before 1930?')

In [ ]:
print('\n=== Q4: before_after  (in_kg=True) ===')
ask('Where did Vladimir Nabokov live after 1960?')

In [ ]:
print('\n=== Q5: first_last  (in_kg=False — expects NULL) ===')
ask('What was the first novel of Vladimir Nabokov?')

In [ ]:
print('\n=== Q6: time_join  (in_kg=True — key hallucination test) ===')
# v2 answer: 'Lyndon B. Johnson' from parametric memory (hallucination)
# v3 expected: Q9640=LBJ from KG via temporal_bucket (Q30 P6 Q9640 1963-1969)
ask('Who was the head of government of the United States when Nabokov was nominated for Nobel Prize in Literature?')

In [ ]:
print('\n=== Q7: relation  (in_kg=True) ===')
ask('What is the relation between Vladimir Nabokov and Vera Nabokova?')

In [ ]:
# Step 5: Define Benchmark Questions (v11)
# 5 Types * 3 Questions = 15 Questions 
# Every type has exactly 1 question with `in_kg=False` (Expected Answer: Unknown/NULL)

BENCHMARK = [
    # --- 1. Simple Entity ---
    {"q": "Who was the spouse of Vladimir Nabokov?", "type": "simple_entity", "gt": "Vera Nabokova", "in_kg": True},
    {"q": "What country is Vladimir Nabokov a citizen of?", "type": "simple_entity", "gt": "United States of America", "in_kg": True},
    {"q": "What was the name of Vladimir Nabokov's pet cat?", "type": "simple_entity", "gt": "Unknown", "in_kg": False},

    # --- 2. Simple Time ---
    {"q": "In what year did Vladimir Nabokov marry Vera Nabokova?", "type": "simple_time", "gt": "1925", "in_kg": True},
    {"q": "When did Vladimir Nabokov reside in Saint Petersburg?", "type": "simple_time", "gt": "1899 - 1917", "in_kg": True},
    {"q": "When did Vladimir Nabokov visit the Moon?", "type": "simple_time", "gt": "Unknown", "in_kg": False}, 

    # --- 3. Before/After ---
    {"q": "Where did Vladimir Nabokov live before 1930?", "type": "before_after", "gt": "Berlin / Saint Petersburg", "in_kg": True},
    {"q": "Where did Vladimir Nabokov live after 1960?", "type": "before_after", "gt": "Montreux", "in_kg": True},
    {"q": "Which novel did Vladimir Nabokov publish after 2020?", "type": "before_after", "gt": "Unknown", "in_kg": False}, 

    # --- 4. First/Last ---
    {"q": "What was the first city where Vladimir Nabokov resided?", "type": "first_last", "gt": "Saint Petersburg", "in_kg": True},
    {"q": "What was the last city where Vladimir Nabokov resided?", "type": "first_last", "gt": "Montreux", "in_kg": True},
    {"q": "What was the last novel published by Vladimir Nabokov?", "type": "first_last", "gt": "Unknown", "in_kg": False}, 

    # --- 5. Time Join ---
    {"q": "Who was the head of government of the USA when Vladimir Nabokov lived in Wellesley?", "type": "time_join", "gt": "Franklin D. Roosevelt", "in_kg": True},
    {"q": "Who was the head of government of the USA when Nabokov was nominated for Nobel Prize in Literature?", "type": "time_join", "gt": "Lyndon B. Johnson / Kennedy", "in_kg": True},
    {"q": "Who was the head of government of the USA when Vladimir Nabokov won the Grammy Award?", "type": "time_join", "gt": "Unknown", "in_kg": False}
]


In [ ]:
# Step 6: Full Benchmark Evaluation (v11)
import pandas as pd
from IPython.display import display, Markdown

def check_answer(ans, true_ans, in_kg):
    if not in_kg:
        return 'unknown' in ans.lower() or 'null' in ans.lower()
    if ans == 'Error' or ans == 'Unknown' or ans == 'NULL': return False
    # Simple substring check (e.g. "Mary" in "Mary (Q123)")
    for t in true_ans.split(' / '):
        if t.lower() in ans.lower(): return True
    return False

def compare_ask_vs_base(benchmark):
    results = []
    
    # 1. Run all questions
    for i, b in enumerate(benchmark, 1):
        q = b['q']
        print(f"\n{'#'*80}\n--- Q{i}/{len(benchmark)}: {q} ---")
        
        # We run tests with debug=False for clean logs
        ans_ask = ask(q, top_k=10, anonymize=True, debug=False)
        ans_base = ask_base(q, top_k=10, anonymize=True)
        
        ask_ok = check_answer(ans_ask, b['gt'], b['in_kg'])
        base_ok = check_answer(ans_base, b['gt'], b['in_kg'])
        
        cat = 'BF'
        if ask_ok and base_ok: cat = 'TP'
        elif ask_ok and not base_ok: cat = 'AO'
        elif not ask_ok and base_ok: cat = 'BO'
        
        if not b['in_kg'] and cat == 'TP': cat = 'TN' # Both Correctly Rejected
        if not b['in_kg'] and not ask_ok and not base_ok: cat = 'BF' # Both Failed to Reject
        
        results.append({
            'Question': q,
            'Type': b['type'],
            'In KG': b['in_kg'],
            'Ground Truth': b['gt'],
            'Ask (Adv)': ans_ask,
            'Base (Raw)': ans_base,
            'Category': cat
        })
        
    df = pd.DataFrame(results)
    
    # 2. Overall Summary Matrix
    counts = df['Category'].value_counts()
    tp_count = counts.get('TP', 0) + counts.get('TN', 0)
    ao_count = counts.get('AO', 0)
    bo_count = counts.get('BO', 0)
    bf_count = counts.get('BF', 0)
    
    overall = pd.DataFrame([
        ['Both OK (Matched GT or Correctly Rejected)', tp_count],
        ['Only Ask (Advanced) OK', ao_count],
        ['Only Base (Raw E5) OK', bo_count],
        ['Both Failed / Hallucinated', bf_count]
    ], columns=['Outcome', 'Count'])
    
    # 3. Detailed Table (by Question Type)
    detailed = df.groupby(['Type', 'Category']).size().unstack(fill_value=0)
    
    # Display Results
    display(Markdown("## 1. Overall Summary Matrix"))
    display(overall.style.hide(axis="index"))
    
    display(Markdown("## 2. Detailed Breakdown by Type"))
    display(detailed)
    
    display(Markdown("## 3. Full Log"))
    display(df)
    
    # Save to CSV
    df.to_csv('benchmark_results_v11.csv', index=False)
    print("\nSaved full results to 'benchmark_results_v11.csv'")
    
    return df, overall, detailed



In [ ]:
df_log, df_overall, df_detailed = compare_ask_vs_base(BENCHMARK)
